**Reviewer note:** This notebook is a preserved development experiment. Read `docs/EXPERIMENTS.md` and `docs/METHODOLOGY.md` before execution. Raw/restricted inputs are not included. Historical references to 'test' do not establish prospective independence or validated online causality.

# AIMS: preparazione del dataset finale

Questo notebook costruisce il dataset supervisionato definitivo per la correzione degli offset di puntamento SRT.

Decisioni consolidate:

- vero XEL ricostruito dai comandi `#fivpt#xoffset`;
- esclusione dei tre cross-scan con `azelOffsets` attivo;
- finestra causale di 2 ore, 60 campioni e 295 sensori;
- esclusione delle finestre con salti termici non plausibili;
- target EL verificato contro i file di calibrazione;
- PCA fittata esclusivamente sul training cronologico;
- nessuna feature FEM, solare, HPBW o sigma del fit nel modello finale.

In [ ]:
# 0. Configurazione
from pathlib import Path
import sys
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def find_project_root():
    candidates = [Path.cwd(), *Path.cwd().parents, Path('/content/metrologyml')]
    for candidate in candidates:
        if (candidate / 'src' / 'aims_pipeline.py').exists():
            return candidate
    raise FileNotFoundError('Cartella del progetto AIMS non trovata')

ROOT = find_project_root()
SRC_DIR = ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from aims_pipeline import build_final_feature_dataset, prepare_samples, save_prepared_dataset

RAW_PATH = ROOT / 'data' / 'raw' / 'real' / 'EffectiveDataset_Tseries_with_azimuth.csv'
XEL_AUDIT_PATH = ROOT / 'data' / 'raw' / 'control' / 'control_crosscheck_effective_target_projection.csv'
POINTING_AUDIT_PATH = ROOT / 'data' / 'raw' / 'control' / 'control_crosscheck_pointing_audit.csv'
OUTPUT_CSV = ROOT / 'data' / 'processed' / 'aims_clean_features.csv'
FEATURE_ARTIFACT = ROOT / 'artifacts' / 'feature_builders.joblib'
REPORT_PATH = ROOT / 'data' / 'processed' / 'aims_clean_features_report.json'
FIGURE_DIR = ROOT / 'outputs' / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# 1. Costruzione delle finestre pulite
samples = prepare_samples(RAW_PATH, XEL_AUDIT_PATH, POINTING_AUDIT_PATH)
audit_summary_df = pd.DataFrame([{
    'campioni_validi': len(samples.metadata),
    'sensori': len(samples.sensor_cols),
    'sequenza_step': samples.sequences.shape[1],
    'train': int(samples.metadata['Split'].eq('train').sum()),
    'validation': int(samples.metadata['Split'].eq('validation').sum()),
    'test': int(samples.metadata['Split'].eq('test').sum()),
    **{f'scarti_{key}': value for key, value in samples.skip_counts.items()},
}])
display(audit_summary_df)
display(samples.metadata.groupby(['Split', 'ReceiverBand'], observed=True).size().rename('count').reset_index())

In [ ]:
# 2. Feature finali leak-free ed esportazione
final_dataset, feature_artifacts = build_final_feature_dataset(samples)
report = save_prepared_dataset(
    final_dataset, feature_artifacts, samples.skip_counts,
    OUTPUT_CSV, FEATURE_ARTIFACT, REPORT_PATH,
)
display(pd.DataFrame([report]))
print('CSV:', OUTPUT_CSV)
print('Trasformatore:', FEATURE_ARTIFACT)

In [ ]:
# 3. Controlli sul dataset modellistico
feature_columns = [column for column in final_dataset if column.startswith('feature__')]
quality_df = pd.DataFrame([{
    'righe': len(final_dataset),
    'feature_usate': len(feature_columns),
    'NaN_nelle_feature': int(final_dataset[feature_columns].isna().sum().sum()),
    'duplicati_timestamp': int(final_dataset['Timestamp'].duplicated().sum()),
    'XEL_target_NaN': int(final_dataset['target_XEL_arcsec'].isna().sum()),
    'EL_target_NaN': int(final_dataset['target_EL_arcsec'].isna().sum()),
}])
display(quality_df)
display(final_dataset[['Timestamp', 'Split', 'Elevation', 'Azimuth', 'ReceiverBand',
                       'target_XEL_arcsec', 'target_EL_arcsec']].head())

In [ ]:
# 4. Figura tesi: copertura temporale e split
colors = {'train': '#0072B2', 'validation': '#E69F00', 'test': '#D55E00'}
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
for axis_name, target_column, ax in [
    ('XEL', 'target_XEL_arcsec', axes[0]),
    ('EL', 'target_EL_arcsec', axes[1]),
]:
    for split_name, group in final_dataset.groupby('Split', sort=False):
        ax.scatter(group['Timestamp'], group[target_column], s=12, alpha=.7,
                   color=colors[split_name], label=split_name)
    ax.axhline(0, color='black', lw=.8)
    ax.set_ylabel(f'{axis_name} [arcsec]')
    ax.grid(True, alpha=.25)
axes[0].set_title('Quality-controlled operational dataset and chronological split')
axes[0].legend(ncol=3)
axes[1].set_xlabel('Time [UTC]')
fig.tight_layout()
fig.savefig(FIGURE_DIR / '00_dataset_timeline_and_splits.png', dpi=300, bbox_inches='tight')
fig.savefig(FIGURE_DIR / '00_dataset_timeline_and_splits.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# 5. Figura tesi: distribuzioni dei target e ricevitori
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].hist(final_dataset['target_XEL_arcsec'], bins=40, color='#0072B2', alpha=.8)
axes[0].set_title('Corrected XEL target'); axes[0].set_xlabel('arcsec'); axes[0].set_ylabel('Samples')
axes[1].hist(final_dataset['target_EL_arcsec'], bins=40, color='#D55E00', alpha=.8)
axes[1].set_title('EL target'); axes[1].set_xlabel('arcsec')
receiver_counts = final_dataset['ReceiverBand'].value_counts().reindex(['K', 'C_high', 'C_low', 'Q']).dropna()
axes[2].bar(receiver_counts.index, receiver_counts.values, color='#009E73')
axes[2].set_title('Samples by receiver band'); axes[2].set_ylabel('Samples')
for ax in axes: ax.grid(True, alpha=.2)
fig.tight_layout()
fig.savefig(FIGURE_DIR / '00_target_and_receiver_distributions.png', dpi=300, bbox_inches='tight')
fig.savefig(FIGURE_DIR / '00_target_and_receiver_distributions.pdf', bbox_inches='tight')
plt.show()

## Output

Il CSV contiene una riga per cross-scan e 43 feature operative condivise dai due rami. Le 295 temperature originali non sono duplicate nel CSV: il trasformatore PCA salvato in `artifacts/feature_builders.joblib` permette di applicare esattamente la stessa trasformazione a nuove finestre.